# requires-grad-propagation — faded example 3: Extend the any-Input Scan to Include kwargs Values (Faded)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `requires-grad-propagation`. Running the beacon reports progress on the `Backprop: requires_grad propagation` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: requires_grad propagation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`requires-grad-propagation`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "requires-grad-propagation"
DD_SUBTOPIC = "Backprop: requires_grad propagation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Some PyTorch-style operations receive tensor inputs as keyword arguments (e.g., `conv2d(x, weight=W, bias=b)`). The `requires_grad` scan must check both `args` and `kwargs.values()` to correctly detect tracked inputs. If `x` has no gradient but `W` does, the output should still propagate gradients. The two scans are OR-combined.

## Faded exercise 3

Implement `propagate_requires_grad_full(args, kwargs, is_differentiable, grad_tracking_enabled)`. The scan over `args` is provided. Your task is to **also scan `kwargs.values()`** and OR the result with the args scan.

Use the same `isinstance(v, t.Tensor) and v.requires_grad` pattern for kwargs values.

**Fill in:** Compute kwarg_tracked by scanning kwargs.values() for tensors with requires_grad=True, then OR it with any_tracked.

In [ ]:
import torch as t

def propagate_requires_grad_full(
    args: tuple,
    kwargs: dict,
    is_differentiable: bool,
    grad_tracking_enabled: bool,
) -> bool:
    any_tracked = any(isinstance(a, t.Tensor) and a.requires_grad for a in args)
    kwarg_tracked = None  # TODO: Compute kwarg_tracked by scanning kwargs.values() for tensors with requires_grad=True, then OR it with any_tracked.
    return grad_tracking_enabled and is_differentiable and (any_tracked or kwarg_tracked)


def _test():
    import torch as t
    tracked = t.tensor([1.0], requires_grad=True)
    untracked = t.tensor([2.0])
    # Tracked in kwargs only -> True
    assert propagate_requires_grad_full((), {'w': tracked}, True, True) == True
    # Tracked in args only -> True
    assert propagate_requires_grad_full((tracked,), {}, True, True) == True
    # Neither args nor kwargs tracked -> False
    assert propagate_requires_grad_full((untracked,), {'b': untracked}, True, True) == False
    # Tracking off -> False even with tracked inputs
    assert propagate_requires_grad_full((tracked,), {'w': tracked}, True, False) == False


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def propagate_requires_grad_full(
    args: tuple,
    kwargs: dict,
    is_differentiable: bool,
    grad_tracking_enabled: bool,
) -> bool:
    any_tracked = any(isinstance(a, t.Tensor) and a.requires_grad for a in args)
    kwarg_tracked = any(isinstance(v, t.Tensor) and v.requires_grad for v in kwargs.values())
    return grad_tracking_enabled and is_differentiable and (any_tracked or kwarg_tracked)
```
</details>